### Reading data from silver layer

In [0]:
df= spark.read.format('delta').load('/Volumes/workspace/default/practice/SuperStore_Sales/Silver')


In [0]:
display(df)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import *

### Data Modelling (Star Schema)

#### creating customer dimension

In [0]:
from pyspark.sql.window import Window

In [0]:
df_cust_dim = df.select(
    "customer_name",
    "segment",
    "country",
    "state"
).dropDuplicates(["customer_name"])
df_cust_dim=df_cust_dim.withColumn('customer_id',row_number().over(Window.orderBy('customer_name')))

df_cust_dim=df_cust_dim.select('customer_id','customer_name','segment','country','state').sort('customer_id')


display(df_cust_dim)


In [0]:
from pyspark.sql.types import *

In [0]:
df_cust_dim.groupBy('customer_name').agg(count("*").alias("count")).filter(col("count")>1).show()

In [0]:
df_cust_dim.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/practice/SuperStore_Sales/Gold/customer_dim")

#### Creating product dimension

In [0]:
df_prod_dim=df.select("product_id","category","sub_category","product_name").distinct().dropDuplicates(["product_id"])
display(df_prod_dim)


In [0]:
df_prod_dim.groupBy('product_id').agg(count("*").alias("count")).filter(col("count")>1).display()

In [0]:
df_prod_dim.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/practice/SuperStore_Sales/Gold/product_dim")

#### creating market dimension

In [0]:
df_market_dim = df.select(
    "market",
    "region",
).dropDuplicates(["market","region"])
df_market_dim=df_market_dim.withColumn('market_id',row_number().over(Window.orderBy('market','region')))

df_market_dim=df_market_dim.select('market_id','market','region').sort('market','region')


display(df_market_dim)


In [0]:
df_prod_dim.write.format('delta').mode('overwrite').save('/Volumes/workspace/default/practice/SuperStore_Sales/Gold/market_dim')

#### Creating order dimension

In [0]:
df_order_dim = df.select(
    "order_id",
    "order_priority",
    "ship_mode"
    
).dropDuplicates(["order_id"])

df_order_dim=df_order_dim.select('order_id','order_priority',"ship_mode").distinct()


display(df_order_dim)

In [0]:
df_order_dim.groupBy('order_id','order_priority','ship_mode').agg(count("*").alias("count")).filter(col("count")>1).show()

In [0]:
df_order_dim.count()

In [0]:
df_order_dim.groupBy('order_id').agg(count("*").alias("count")).filter(col("count")>1).show()

In [0]:
df_order_dim.write.format('delta').mode('overwrite').save('/Volumes/workspace/default/practice/SuperStore_Sales/Gold/order_dim')

In [0]:
df_market_dim = df_market_dim.withColumn("market", upper(trim(col("market")))) \
                             .withColumn("region", upper(trim(col("region"))))

# Clean raw data
df_clean = df.withColumn("market", upper(trim(col("market")))) \
             .withColumn("region", upper(trim(col("region"))))

df_sales_fact = df_clean \
    .join(df_cust_dim, "customer_name", "left") \
    .join(df_prod_dim, "product_id", "left") \
    .join(df_market_dim, ["market", "region"], "left") \
    .join(df_order_dim, "order_id", "left")

display(df_sales_fact)   
        


In [0]:
df.count()


In [0]:
df_sales_fact.count()

In [0]:
df_sales_fact=df_sales_fact.select('order_id','customer_id','order_date','ship_date','product_id','market_id','sales','quantity','discount','profit', 'shipping_cost')
display(df_sales_fact)
df_sales_fact.count()



In [0]:
display(df_market_dim)

In [0]:
df_sales_fact.filter(col('market_id').isNull()).count()

#### Validate Data 

In [0]:

# Customer dimension PK
df_cust_dim.groupBy("customer_id").agg(count("*").alias("cnt")).filter(col("cnt")>1).show()

# Product dimension PK
df_prod_dim.groupBy("product_id").agg(count("*").alias("cnt")).filter(col("cnt")>1).show()

# Market dimension PK
df_market_dim.groupBy("market_id").agg(count("*").alias("cnt")).filter(col("cnt")>1).show()

# Order dimension PK
df_order_dim.groupBy("order_id").agg(count("*").alias("cnt")).filter(col("cnt")>1).show()

In [0]:
df_sales_fact.join(df_cust_dim, "customer_id", "left_anti").count() 

df_sales_fact.join(df_prod_dim, "product_id", "left_anti").count() 

df_sales_fact.join(df_market_dim, "market_id", "left_anti").count() 

df_sales_fact.join(df_order_dim, "order_id", "left_anti").count() 

In [0]:
df.agg(sum(col('sales')).alias('total_sales_df')).show()
df_sales_fact.agg(sum(col('sales')).alias('total_sales_df_sales_fact')).show()

In [0]:
df.groupBy('product_id').agg(count(col('quantity')).alias('total_quantity_df')).show()
df_sales_fact.groupBy('product_id').agg(count(col('quantity')).alias('total_quantity_df_sales_fact')).show()

In [0]:
df.agg(count(col('quantity')).alias('total_quantity_df')).show()
df_sales_fact.agg(count(col('quantity')).alias('total_quantity_df_sales_fact')).show()

df.select(sum("quantity")).show()
df_sales_fact.select(sum("quantity")).show()

In [0]:
df_order_dim.groupBy('order_id').agg(count(col('order_id')).alias('count')).filter(col('count')>1).show()
df.groupBy("order_id",
"product_id",
"customer_name",
"order_date",
"ship_date") \
    .agg(count("*").alias("count")) \
    .filter(col("count") > 1) \
    .show()

In [0]:
df.groupBy("order_id") \
.agg(countDistinct("ship_mode").alias("ship_modes")) \
.filter(col("ship_modes") > 1) \
.show()

In [0]:
df.groupBy("order_id").agg(countDistinct("ship_mode").alias("count")).filter(col("count")>1).show()

In [0]:
df_date_dim=df.select(col('order_date').alias('date')).union\
    (df.select(col('ship_date')).alias('date')).distinct()
display(df_date_dim)

In [0]:
df_date_dim=df_date_dim.sort(col('date'))
display(df_date_dim)

In [0]:
df_date_dim = df_date_dim.withColumn("year", year("date")) \
    .withColumn("quarter", quarter("date")) \
    .withColumn("month", month("date")) \
    .withColumn("month_name", date_format("date","MMMM")) \
    .withColumn("week_of_year", weekofyear("date")) \
    .withColumn("day", dayofmonth("date")) \
    .withColumn("day_name", date_format("date","EEEE"))

display(df_date_dim)

In [0]:
df_date_dim=df_date_dim.withColumn('date_id',
                                   date_format('date','yyyyMMdd').cast("int")
                                   ).select("date_id","date","year","quarter","month","month_name","week_of_year","day","day_name").orderBy("date")
display(df_date_dim)


In [0]:
display(df_sales_fact)

In [0]:
df_order_date_dim=df_date_dim.select(col('date').alias('order_date_dim_date'), col('date_id').alias('date_id_order_date'))

df_sales_fact=df_sales_fact.join(df_order_date_dim,df_sales_fact['order_date']==df_order_date_dim['order_date_dim_date'],'left').drop('order_date_dim_date')


In [0]:
display(df_sales_fact)

In [0]:
df_ship_date_dim=df_date_dim.select(col('date').alias('ship_date_dim_date'), col('date_id').alias('date_id_ship_date'))

df_sales_fact=df_sales_fact.join(df_ship_date_dim,df_sales_fact['ship_date']==df_ship_date_dim['ship_date_dim_date'],'left').drop('ship_date_dim_date')

In [0]:
display(df_sales_fact)

In [0]:
df_date_dim.write.format('delta').mode('overwrite').save('/Volumes/workspace/default/practice/SuperStore_Sales/Gold/date_dim/')

In [0]:
df_sales_fact.write.format('delta').mode('overwrite').option('overwriteSchema','true').save('/Volumes/workspace/default/practice/SuperStore_Sales/Gold/sales_fact/')

In [0]:
df_market_dim.write.format('delta').mode('overwrite').option('overWriteSchema', 'true').save('/Volumes/workspace/default/practice/SuperStore_Sales/Gold/market_dim/')
display(df_market_dim)